In [1]:
!pip install deepeval groq json-repair huggingface_hub langfuse ollama >> /dev/null

In [4]:
!sudo apt-get update && sudo apt-get install -y zstd >> /dev/null

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,947 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,

In [5]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Retrieval

In [ ]:
import os
import re
import time
import json
import llama_cpp
import subprocess
import numpy as np

from tqdm import tqdm
from llama_cpp import Llama
from langfuse import Langfuse
from huggingface_hub import hf_hub_download
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import (ContextualPrecisionMetric,
                              ContextualRecallMetric,
                              ContextualRelevancyMetric,
                              AnswerRelevancyMetric,
                              FaithfulnessMetric,
                              GEval)

from src.retrieval.query import retrieve
from src.retrieval.config import LOCAL_RERANK
from src.generation.generator import generate_response
from src.retrieval.reranker import load_local_reranker
from src.retrieval.chunk_utils import merge_ranked_chunks
from src.retrieval.vector_store import get_pinecone_index
from src.retrieval.embedder import (get_embeddings, load_dense_embedding_model,
                                    load_sparse_embedding_model)

In [ ]:
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY')
LANGFUSE_BASE_URL   = os.getenv("LANGFUSE_BASE_URL")

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_BASE_URL
)

In [ ]:
with open('/content/enterprise-rag/questions.json', 'r', encoding='utf-8') as f:
  questions = json.load(f)

In [ ]:
def initialize_search_pipeline():
    pc, pc_index = get_pinecone_index()
    dense_tokenizer, dense_model = load_dense_embedding_model()
    sparse_tokenizer, sparse_model, sparse_input_names = load_sparse_embedding_model()
    reranker = load_local_reranker() if LOCAL_RERANK else None

    return (pc, pc_index, dense_tokenizer, dense_model,
            sparse_tokenizer, sparse_model,
            sparse_input_names, reranker)

(pc, pc_index, dense_tokenizer, dense_model,
  sparse_tokenizer, sparse_model, sparse_input_names,
  reranker) = initialize_search_pipeline()


dense_model.use_io_binding=False
reranker[1].use_io_binding=False

In [ ]:
def run_query(user_name, query, pc, pc_index, 
            dense_tokenizer, dense_model, 
            sparse_tokenizer, sparse_model,
            sparse_input_names, reranker_model) -> tuple:
    

    query_dense_embedding, query_sparse_embedding = get_embeddings(
        dense_tokenizer, 
        dense_model, 
        sparse_tokenizer,
        sparse_model,
        sparse_input_names,
        query
    )


    retrieved_docs = retrieve(
      user_name=user_name,
      query=query,
      query_dense_embedding=query_dense_embedding,
      query_sparse_embedding=query_sparse_embedding,
      pc_index=pc_index,
      reranker=reranker_model
    )

    merged_docs = merge_ranked_chunks(retrieved_docs.data)
    sources = [doc['id'] for doc in merged_docs]
    
    return merged_docs, sources

In [ ]:
results = []
user_name = None
for question in tqdm(questions):
  if question['question_type'] == 'basic':
    (merged_docs, sources) = run_query(user_name, question['question'], pc, pc_index, 
                              dense_tokenizer, dense_model, 
                              sparse_tokenizer, sparse_model, 
                              sparse_input_names, reranker)
  
  
    results.append({'question_data': question, 'response_data': merged_docs})

In [ ]:
with open("final_retreival_results.jsonl", "w", encoding="utf-8") as f:
    for item in merged_data:
        f.write(json.dumps(item) + "\n")

# Generation

In [ ]:
with open('/content/enterprise-rag/final_retreival_results.jsonl', 'r', encoding='utf-8') as f:
  data = [json.loads(line) for line in f]

In [ ]:
SYSTEM_PROMPT = langfuse.get_prompt('generation_system_prompt', label='production').prompt

In [ ]:
os.environ["GGML_CUDA_DISABLE_GRAPHS"] = "1"

def build_context_block(query: str, merged_docs: list[dict]) -> str:
    context = f"Context:"
    for doc in merged_docs:
        source = doc.get("id", "")
        text = doc.get("text", "")
        chunk_range = doc.get("chunk_range")
        label = f"Document: {source} (chunk {chunk_range})" if chunk_range else f"Document: {source}"
        context += f"\n{label}\n{text}\n{'=' * 20}\n"

    context += f"Query : {query}"
    return context

def ensure_model_ready():
    os.makedirs(MODEL_DIR, exist_ok=True)
    model_path = os.path.join(MODEL_DIR, MODEL_FILENAME)
    if not os.path.exists(model_path):
        hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=MODEL_FILENAME,
            local_dir=MODEL_DIR,
            local_dir_use_symlinks=False
        )
    return model_path


MODEL_DIR = "model_store"
MODEL_FILENAME = "Qwen3-0.6B-Q5_K_M.gguf"
HF_REPO_ID = "unsloth/Qwen3-0.6B-GGUF"

model_path = ensure_model_ready()

llm = Llama(
    model_path   = model_path,
    n_ctx        = 4096,
    n_threads    = MAX_THREADS
    n_batch      = 2048,
    n_gpu_layers = -1,
    mmap         = True,
    verbose      = False,
    chat_format  = "chatml"
)

llm.create_chat_completion(
    messages=[{"role": "user", "content": "init"}],
    max_tokens=1,
    stream=False
)

In [ ]:
def generate(query: str, merged_docs: list[dict]) -> str:
    context = build_context_block(query, merged_docs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": context}
    ]
    response = llm.create_chat_completion(
      messages=messages,
      max_tokens=2058,
      temperature=0.0,
      stream=False
    )

    generated_text = response["choices"][0]["message"]["content"]
    total_tokens = response['usage']['total_tokens']
    finish_reason = response['choices'][0]['finish_reason']
    
    return generated_text, total_tokens, finish_reason

In [ ]:
for i, d in tqdm(enumerate(data), total=len(data)):
  query = d['question_data']['question']
  docs  = d['response_data']

  context = build_context_block(query, docs)
  response, total_tokens, finish_reason = generate(query, docs)

  data[i]['answer_data'] = {
      'answer': response,
      'total_tokens': total_tokens,
      'finish_reason' : finish_reason
  }

In [ ]:
with open("final_response_data.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Evaluation

In [2]:
with open('/content/enterprise-rag/final_response_data.jsonl', 'r', encoding='utf-8') as f:
  data = [json.loads(line) for line in f]

In [6]:
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + env.get("LD_LIBRARY_PATH", "")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    env=env, # Pass the GPU environment variables
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

In [ ]:
# !ollama pull deepseek-r1:1.5b
# !ollama pull llama3.1:8b
# !ollama pull llama3.1:8b-instruct-q5_K_M
!ollama pull deepseek-r1:7b-qwen-distill-q4_K_M

In [12]:
from deepeval.models import OllamaModel

ollama_model = OllamaModel(
    # model="deepseek-r1:1.5b",
    # model = "llama3.1:8b-instruct-q5_K_M",
    model = "deepseek-r1:7b-qwen-distill-q4_K_M",
    temperature=0.0
)

In [44]:
np.random.seed(69)
eval_model = ollama_model

contextual_recall = ContextualRecallMetric(model=eval_model)
faithufullness = FaithfulnessMetric(model=eval_model)
answer_relevancy = AnswerRelevancyMetric(model=eval_model)
answer_correctness = GEval(
    name="Answer Correctness",
    model=eval_model,
    evaluation_params=[
        SingleTurnParams.EXPECTED_OUTPUT,
        SingleTurnParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "Determine whether the actual output is correct based on the expected output."
    ]
)

In [ ]:
test_cases = []
selected_test_cases = np.random.choice(data, 20, replace=False)

for i in range(len(selected_test_cases)):
  answer = re.sub(r'<think>.*?</think>', '',
            selected_test_cases[i]['answer_data']['answer'],
            flags=re.DOTALL).strip()
  answer = re.sub(r'<[^>]*>', '', answer).strip().replace('Answer:', " ").split('Sources:')[0].strip()

  test_cases.append(LLMTestCase(
      input = selected_test_cases[i]['question_data']['question'],
      actual_output = answer,
      expected_output = data[i]['question_data']['gold_answer'],
      retrieval_context=['.'.join(resp['text'].strip().replace('\\n', '\n').split('\n')) for resp in selected_test_cases[i]['response_data']]
  ))

In [ ]:
contextual_recall_score = []
answer_correctness_score = []
faithufullness_score = []
answer_relevancy_score = []

count = 0

total_steps = len(test_cases) * 4  

with tqdm(total=total_steps, desc="Evaluating Metrics") as pbar:
    for test_case in test_cases:
        try:
            contextual_recall.measure(test_case)
            pbar.update(1)

            answer_correctness.measure(test_case)
            pbar.update(1) 

            answer_relevancy.measure(test_case)
            pbar.update(1) 

            faithufullness.measure(test_case)
            pbar.update(1)

            contextual_recall_score.append(contextual_recall.score)
            answer_correctness_score.append(answer_correctness.score)
            answer_relevancy_score.append(answer_relevancy.score)
            faithufullness_score.append(faithufullness.score)
            
            count += 1

        except Exception as err:
            raise


In [72]:
def calculate_batch_recall_at_k(dataset, k):
    total_recall = 0.0
    
    for result in dataset:
        raw_path = result['question_data']['expected_doc_ids'][0].split('/')
        expected_doc = raw_path[0] + '_' + raw_path[-1].split('.json')[0]
        
        sorted_responses = sorted(
            result['response_data'], 
            key=lambda x: x['score'], 
            reverse=True
        )
        
        seen_ids = set()
        deduplicated_ids = []
        for item in sorted_responses:
            item_id = item['id']
            if item_id not in seen_ids:
                seen_ids.add(item_id)
                deduplicated_ids.append(item_id)
                
        if expected_doc in deduplicated_ids[:k]:
            total_recall += 1.0

    return round(total_recall / len(dataset), 3)


def calculate_batch_mrr(dataset):
    total_mrr = 0.0
    
    for result in dataset:
        raw_path = result['question_data']['expected_doc_ids'][0].split('/')
        expected_doc = raw_path[0] + '_' + raw_path[-1].split('.json')[0]
        
        sorted_responses = sorted(
            result['response_data'], 
            key=lambda x: x['score'], 
            reverse=True
        )
        
        seen_ids = set()
        deduplicated_ids = []
        for item in sorted_responses:
            item_id = item['id']
            if item_id not in seen_ids:
                seen_ids.add(item_id)
                deduplicated_ids.append(item_id)
                
        if expected_doc in deduplicated_ids:
            rank = deduplicated_ids.index(expected_doc) + 1
            total_mrr += 1.0 / rank

    return round(total_mrr / len(dataset), 3)

def compute_statistical_metrics(results) -> dict:
    return {
        "recall@5":  calculate_batch_recall_at_k(results, k=5),
        "recall@10": calculate_batch_recall_at_k(results, k=10),
        "mrr":       calculate_batch_mrr(results),
    }

# Log to Langfuse

In [73]:
metrics = compute_statistical_metrics(data)
metrics = {**metrics, 
           'context_recall'    : round(np.mean(contextual_recall_score), 3),
           'answer_correctness': round(np.mean(answer_correctness_score), 3),
           'answer_relevancy'  : round(np.mean(answer_relevancy_score), 3),
           'faithufullness'    : round(np.mean(faithufullness_score), 3),
          }

{'recall@5': 0.886,
 'recall@10': 0.886,
 'mrr': 0.88,
 'context_recall': 0.772,
 'answer_correctness': 0.742,
 'answer_relevancy': 0.845,
 'faithufullness': 0.937}

In [75]:
def log_scores_to_langfuse(metrics):
    with langfuse.start_as_current_observation(name="deepeval_evaluation", as_type="span") as span:
        for name, value in metrics.items():
            span.score(name=name, value=value)

In [76]:
log_scores_to_langfuse(metrics)